# Settings

In [1]:
parameters_file_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/params.json"
convert_complex = True
apply_directional_filter = False     #!convert_complex must be True for this
directional_filter_direction = "up"  # up, down
apply_hp_svd = False

#Simulation settings
using_simulation_data = False
rf_matrix_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/rf0001.mat"

#Real data settings
using_matlab_data = True       #! high RAM usage for big files -> convert mat file to npy for memory map usage and set using_simulation_data to False
#matlab_rf_data_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/rf0001.mat"
interpolate_data = True

#Batch processing settings
batch_processing = True
batch_size = 1                 #ignored if batch_processing is False
frames_to_process = 160
MAX_WORKERS = 10                 #max is 12 for the current computer (10 max recommended)

#Saving parameters
rf_data_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/"
folder_name = "mouse_test_classes"
processed_rf_matrix_name = "001_rf_IQ"


# Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import timeit
import os
import math
import concurrent.futures
from numba import njit
from scipy.signal import firwin, hilbert, butter, lfilter
from scipy.io import loadmat

import modules.filters as filters
import modules.dsp as dsp
import modules.preprocessing as pp

# Import parameters

In [3]:
def load_parameters(json_file_path):
    with open(json_file_path, 'r') as f:
        params = json.load(f)
    return params


In [4]:
def add_extra_params(params):
    
    params["rf"]["upsampled_samples"] = params["rf"]["samples"] + (params["rf"]["interpolation_factor"] - 1) * (params["rf"]["samples"] - 1)
    params["rf"]["upsampled_sampling_rate"] = params["rf"]["sampling_rate"] * params["rf"]["interpolation_factor"] 

    return params

In [5]:
params = load_parameters(parameters_file_path)
params = add_extra_params(params)

pos = np.linspace(start=0, stop=params["array"]["length"], num=params["array"]["length"], dtype=int) #array position sampling
time = np.linspace(start=0, stop=1 / params["rf"]["sampling_rate"] * params["rf"]["samples"], num=params["rf"]["samples"]) #time sampling
C = params["rf"]["C"] #speed of sound

print(f"Max acquisition depth : {params["array"]["acq_depth"][1]} m") #v = d / t => d = v * t / 2 (round trip) with t = sample_step * samples

Max acquisition depth : 0.012 m


# Create workspace

In [6]:
if not os.path.exists(f"{rf_data_path}{folder_name}"):
    os.makedirs(f"{rf_data_path}{folder_name}")
    print(f"Folder {rf_data_path}{folder_name} doesn't exist. Setting up the folder.")

# Run multiprocessing 

In [7]:
preprocessing = pp.Preprocessing(
    params=params, 

    MAX_WORKERS=MAX_WORKERS, 
    frames_to_process=frames_to_process, 
    batch_size=batch_size,

    rf_matrix_path=rf_matrix_path, 
    using_simulation_data=using_simulation_data, 
    using_matlab_data=using_matlab_data, 
    convert_complex=convert_complex,
    interpolate_data=interpolate_data,

    apply_hp_svd=apply_hp_svd,
    apply_directional_filter=apply_directional_filter,
    directional_filter_direction=directional_filter_direction,

    rf_data_path=rf_data_path,
    folder_name=folder_name,
    processed_rf_matrix_name=processed_rf_matrix_name,
    )

preprocessing.run_preprocessing()

Memmap file with shape: (1535, 128, 16, 160) saved at: /home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/mouse_test_classes/001_rf_IQ.npy

Starting process pool with 10 workers.
Total frames to process: 160
Batch size: 1
Total batches: 160
Last batch identified: Start Frame 159, Size 1

Waiting for all batches to complete...

Processing batch 5 [5/160] (Process ID: 58238)
  • Processing from frame 4 to frame 5
  • Processing 1 frame(s)
  • RF matrix: Imported RF matrix with shape: (768, 128, 16, 1)

Processing batch 3 [3/160] (Process ID: 58234)
  • Processing from frame 2 to frame 3
  • Processing 1 frame(s)
  • RF matrix: Imported RF matrix with shape: (768, 128, 16, 1)

Processing batch 6 [6/160] (Process ID: 58242)
  • Processing from frame 5 to frame 6
  • Processing 1 frame(s)
  • RF matrix: Imported RF matrix with shape: (768, 128, 16, 1)

Processing batch 9 [9/160] (Process ID: 58245)
  • Processing from frame 8 to frame 9
  • Processing 1 frame(s)
  • RF matrix